# 03 · Iteration & Laziness — **Depth**

> **Pull this notebook when:** you build token streaming (Week 9) or any data flow that shouldn't
> load everything into memory at once. Laziness is the backbone.

Depth here is the **iterator protocol** under `for`, and what "lazy" actually means — observed, not
asserted. Predict before running.

---

## 3.1 — Predict: a generator is one-shot

```python
gen = (x for x in range(3))
first = list(gen)
second = list(gen)
```
What are `first` and `second`?

In [ ]:
gen = (x for x in range(3))
first = list(gen)
second = list(gen)
print(first)    # ?
print(second)   # ?

<details>
<summary>▶ Reveal</summary>

`first == [0, 1, 2]`, `second == []`.

A generator is an **iterator**, and iterators are *exhaustible* — once consumed, they're spent. The
first `list(gen)` drained it to `StopIteration`; the second finds nothing left. A list, by contrast,
is *iterable* but not itself an iterator — `iter(list)` makes a fresh iterator each time, so you can
loop it repeatedly. The distinction (iterable vs iterator) is the next exercise.

</details>

## 3.2 — Predict: iterable vs iterator identity

`iter(x)` returns an iterator for `x`. Predict whether each `iter(...)` returns the *same* object back.

```python
lst = [1, 2, 3]
gen = (i for i in range(3))
print(iter(lst) is lst)    # ?
print(iter(gen) is gen)    # ?
```

In [ ]:
lst = [1, 2, 3]
gen = (i for i in range(3))
print(iter(lst) is lst)
print(iter(gen) is gen)

<details>
<summary>▶ Reveal</summary>

`False`, then `True`.

A **list is iterable but not an iterator**: `iter(lst)` builds a *new* list-iterator object each call
(so you can have several independent loops over the same list). A **generator is its own iterator**:
`iter(gen) is gen` — calling `iter` on it returns itself, which is why it's one-shot. This is the
formal version of 3.1: "iterable" means "can produce an iterator via `__iter__`"; "iterator" means
"produces values via `__next__` and is consumed as it goes."

That's exactly what `for x in thing` does under the hood: call `iter(thing)` once to get an iterator,
then call `next(...)` on it until `StopIteration`.

</details>

## 3.3 — Build the same thing two ways

Implement `squares_up_to(n)` — the squares `1, 4, 9, ..., n²` — **twice**: once as a generator function
(`squares_gen`), once as a class implementing the iterator protocol by hand (`SquaresIter`). They
should behave identically. This makes concrete that a generator *is* a hand-written iterator with the
`__next__`/`StopIteration` bookkeeping automated.

In [ ]:
def squares_gen(n):
    # YOUR CODE HERE — generator with yield
    pass

class SquaresIter:
    def __init__(self, n):
        self.n = n
    def __iter__(self):
        # YOUR CODE HERE
        pass
    def __next__(self):
        # YOUR CODE HERE
        pass

# Tests
assert list(squares_gen(4)) == [1, 4, 9, 16]
assert list(SquaresIter(4)) == [1, 4, 9, 16]
assert list(squares_gen(0)) == []
assert list(SquaresIter(0)) == []
print("3.3 passed")

<details>
<summary>▶ The equivalence</summary>

The generator's `yield` does, automatically, what the class does by hand: pause-and-resume with state
preserved (`i`), and raise `StopIteration` when the loop ends (the generator does this when its
function body returns). The class spells out the three protocol pieces — `__iter__` sets up state and
returns the iterator, `__next__` produces one value or raises `StopIteration`. A generator is the same
machine with the boilerplate hidden. Reach for a generator unless you need the object to carry extra
methods or be re-startable.

</details>

## 3.4 — Predict: when does the work actually happen?

This is the heart of laziness. A logging function records when it runs. Predict the **order** of the
printed log relative to the `print("---")` lines.

```python
def loud(x):
    print(f"  computing {x}")
    return x * 10

nums = [1, 2, 3]
pipeline = (loud(n) for n in nums)   # generator expression — nothing computed yet
print("--- built, not consumed ---")
first = next(pipeline)
print("--- pulled one ---")
rest = list(pipeline)
print("--- pulled rest ---")
```

In [ ]:
def loud(x):
    print(f"  computing {x}")
    return x * 10

nums = [1, 2, 3]
pipeline = (loud(n) for n in nums)
print("--- built, not consumed ---")
first = next(pipeline)
print("--- pulled one ---")
rest = list(pipeline)
print("--- pulled rest ---")

<details>
<summary>▶ Reveal</summary>

The log appears like this:

```
--- built, not consumed ---
  computing 1
--- pulled one ---
  computing 2
  computing 3
--- pulled rest ---
```

Building the generator expression computes **nothing** — no `computing` line before "built". The work
happens only when values are *pulled*: `next()` computes exactly one (`computing 1`), then `list()`
pulls the remaining two. This is **lazy, pull-based** evaluation: a generator does work on demand, one
element at a time, not up front. That's why it can stream a huge source without holding it all in
memory — the property you'll lean on for token streaming and large-file processing.

</details>

## 3.5 — Predict: chained generators interleave

Two stacked generators. Predict the log order — does it compute *all* doublings first, then all
increments? Or something else?

```python
def double(seq):
    for x in seq:
        print(f"  double {x}")
        yield x * 2

def inc(seq):
    for x in seq:
        print(f"  inc {x}")
        yield x + 1

result = inc(double([1, 2]))   # chained, both lazy
print("--- consuming ---")
out = list(result)
```

In [ ]:
def double(seq):
    for x in seq:
        print(f"  double {x}")
        yield x * 2

def inc(seq):
    for x in seq:
        print(f"  inc {x}")
        yield x + 1

result = inc(double([1, 2]))
print("--- consuming ---")
out = list(result)
print(out)

<details>
<summary>▶ Reveal</summary>

```
--- consuming ---
  double 1
  inc 2
  double 2
  inc 4
[3, 5]
```

The stages **interleave per element**, not per stage. One value flows all the way through the chain
before the next one starts: `double` produces `2`, `inc` immediately consumes it and produces `3`;
*then* `double` produces the next. Each `inc` pull reaches back through `double` for exactly one value.
That's the defining behavior of a lazy pipeline — elements stream through end-to-end, so the whole
chain only ever holds one item in flight. No intermediate list of all-doubled values is ever built.

</details>

## ★ Milestone 03 — A lazy processing pipeline

Synthesize: build a multi-stage pipeline of generators that processes a source one item at a time,
never materializing the full sequence. Implement three generator stages and a driver:

- `read_records(rows)` — yields each row (simulating reading a large source lazily)
- `parse(records)` — yields `int(row)` for each row
- `keep_even(numbers)` — yields only the even numbers
- `pipeline(rows)` — wires them together and returns the final lazy generator

Then a test confirms it produces the right values **and** stays lazy (pulling one value doesn't process
the whole source). This is the streaming pattern behind Week 9's token output.

(build it below)

In [ ]:
def read_records(rows):
    # YOUR CODE HERE — yield each row
    pass

def parse(records):
    # YOUR CODE HERE — yield int(r) for each
    pass

def keep_even(numbers):
    # YOUR CODE HERE — yield only even numbers
    pass

def pipeline(rows):
    # YOUR CODE HERE — compose the three stages, return the final generator
    pass

# Tests
rows = ["1", "2", "3", "4", "5", "6"]
result = pipeline(rows)

# it's lazy: building it consumed nothing; it's a generator, not a list
import types
assert isinstance(result, types.GeneratorType)

# pulling ONE value should not require processing the whole source
assert next(result) == 2

# the rest streams correctly
assert list(result) == [4, 6]

# full run on fresh input
assert list(pipeline(["10", "11", "12", "13"])) == [10, 12]
assert list(pipeline([])) == []
print("Milestone 03 passed")

<details>
<summary>▶ Why it's lazy end-to-end</summary>

Each stage is a generator that pulls from the previous one on demand. `pipeline` just wires them:
`keep_even(parse(read_records(rows)))`. Nothing runs until you pull from the outer generator — and
each pull threads exactly one item through all three stages (read → parse → filter), as you saw
interleave in 3.5. The `isinstance(result, GeneratorType)` check proves no list was built; the
single-`next` check proves it streams. Swap `rows` for a file object and this processes a multi-GB
file with one record in memory at a time — which is the point.

</details>